<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/891_MSOv3_ReportGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a **very strong report generation layer**. Architecturally, it completes the pipeline you designed earlier:

```
Telemetry → Metrics → Signals → Executive Report
```

This file implements the **final step: translating operational intelligence into an executive briefing**.



---

# Report Generation — Mission Orchestrator V3

The report generation module converts operational telemetry and intelligence metrics into a **structured executive report**.

While the earlier stages of the orchestrator focus on data ingestion and analytical scoring, this module focuses on **communication**.

Operational intelligence only creates value when decision makers can understand it quickly. This module ensures that the results of the orchestrator’s analysis are delivered in a clear, structured, and consistent format suitable for business leaders.

Instead of presenting raw metrics or logs, the system produces a **decision-oriented operational report**.

---

# Role in the Orchestrator Architecture

This module sits at the final stage of the orchestrator pipeline.

The full system flow now looks like:

```
Telemetry Loader
      ↓
Signature Metrics Engine
      ↓
Operational Status Signals
      ↓
Report Generation Layer   ← this module
```

By separating analysis from reporting, the architecture maintains two important qualities:

**Analytical transparency** — metrics are computed independently of presentation.

**Communication clarity** — reporting focuses purely on explaining the results.

This separation allows organizations to adjust report formats without affecting the underlying intelligence logic.

---

# Formatting and Presentation Utilities

Several small helper functions are included to keep the report readable and consistent.

For example:

```
_fmt_pct()
```

Standardizes percentage formatting so metrics appear consistently throughout the report.

Another helper function identifies the most recent mission run:

```
_get_latest_run()
```

This allows the report to include the **latest KPI snapshot**, which provides leaders with a quick view of current mission performance.

These small formatting utilities improve readability without complicating the core reporting logic.

---

# Mission Status Section

The first section of the report communicates the **overall operational signal**.

This section includes:

```
MISSION STATUS
DATA QUALITY
```

The mission status signal summarizes the orchestrator’s intelligence analysis.

Examples include:

```
MISSION STATUS: HEALTHY
MISSION STATUS: AT RISK
MISSION STATUS: OFF TARGET
```

The data quality signal provides context for how reliable the analysis is.

For example:

```
DATA QUALITY: HIGH (8 runs loaded)
```

For executives, these signals serve as the **quickest possible operational summary**. A leader can determine the overall system condition within seconds.

---

# Executive Summary

The Executive Summary translates numerical intelligence metrics into **clear operational language**.

Instead of listing scores directly, the system interprets them using descriptive language such as:

```
highly efficient
moderate friction
stable agents
mostly automated workflow
```

This approach ensures that the report remains accessible to readers who may not be familiar with the scoring system.

For example, a sentence like:

```
Operational friction is moderate (Operational Friction Score: 34).
```

communicates both the **interpretation** and the **underlying metric**.

The summary also includes the most recent KPI snapshot, providing immediate context for mission performance.

This design mirrors how operational briefings are typically structured in executive environments.

---

# Performance Trend Analysis

The trend section provides a simple analysis of mission performance over time.

It evaluates how the mission’s primary KPI has changed across historical runs.

Key values displayed include:

```
Baseline
Latest value
Improvement vs baseline
```

The report also displays the recent KPI values so readers can quickly see the performance trajectory.

Even this simple trend analysis is extremely valuable operationally because it answers the most important strategic question:

**Is the mission getting better over time?**

This question is central to evaluating whether AI-driven workflows are producing real business improvement.

---

# Additional Operational Sections

The report also includes placeholders for additional analysis sections:

```
BOTTLENECK ANALYSIS
RISK ANALYSIS
AGENT PERFORMANCE ANALYSIS
PORTFOLIO HEALTH SUMMARY
```

In the current V3 implementation, these sections intentionally remain simple.

They point to the relevant telemetry datasets rather than attempting to compute complex analyses within the report generator.

This is a deliberate design choice.

The reporting layer should remain **lightweight and deterministic**, while deeper analysis logic can evolve in later versions of the orchestrator.

Keeping the report generator simple helps ensure the system remains predictable and easy to maintain.

---

# Report Assembly

The `assemble_report` function combines the generated sections into a final Markdown document.

The sections are assembled in a consistent order:

```
Mission Status
Executive Summary
Mission Performance Trend
Bottleneck Analysis
Risk Analysis
Agent Performance Analysis
Portfolio Health Summary
```

This structure mirrors the reporting standard defined earlier in the telemetry framework.

Consistency across reports makes it easier for leaders to interpret results across different missions.

---

# Report Storage and Traceability

After assembling the report, the system saves it to disk with a timestamped filename.

Example format:

```
mso_v3_report_M001_20260313_143512.md
```

This naming convention ensures that each report is preserved as an **immutable operational artifact**.

Historical reports can be reviewed later to analyze how mission performance evolved over time.

This kind of traceability is critical in operational systems where leadership may need to audit past decisions or investigate performance changes.

---

# Why This Design Builds Executive Trust

This reporting design reinforces several principles that increase confidence in AI-driven systems.

First, the report is built entirely from **deterministic metrics**, not model-generated interpretations.

Second, the system communicates intelligence using **structured operational language** rather than raw telemetry.

Third, every report is stored as a persistent document, creating a historical record of system behavior.

These characteristics allow leaders to treat the orchestrator as an **operational monitoring system**, rather than an experimental AI tool.

---

# Small Improvements Worth Considering

The implementation is already strong. A few refinements could improve robustness and future extensibility.

---

### Sort Runs by Time

The `_get_latest_run()` function assumes that mission runs are already in chronological order.

To make the system safer, runs could be sorted explicitly by timestamp before selecting the latest run.

Example logic:

```
sort runs by start_time
```

This ensures correct behavior even if telemetry files are appended out of order.

---

### KPI Trend Direction Awareness

The trend calculation currently assumes that improvement means **lower values are better**.

For KPIs where higher values are desirable, the improvement formula should reverse.

The system already tracks:

```
higher_is_better
```

Using that flag would make trend reporting more robust.

---

### Better Placeholder Messaging

The placeholder analysis sections currently direct readers to telemetry files.

You could slightly enhance these sections by summarizing key values from those datasets.

For example:

```
Top bottleneck task: Document Verification
High severity risks detected: 1
Agents with highest escalation rate: A4
```

This would strengthen the executive value of the report without significantly increasing complexity.

---

# Architectural Impact

This module completes the orchestrator’s transformation of telemetry into operational intelligence.

At this stage the pipeline has achieved its final goal:

```
Operational data
→ measurable intelligence metrics
→ executive-ready insights
```

The result is a system that allows leaders to evaluate AI-driven operations in the same way they evaluate traditional business processes.

Instead of asking:

```
What did the agents do?
```

the system answers:

```
Is the mission improving?
Where are the bottlenecks?
Are the agents reliable?
How automated is the workflow?
```

These are the kinds of questions that determine whether AI systems produce real operational value.

---

# Overall Evaluation

This report generation module demonstrates strong architectural discipline.

Key strengths include:

• clear separation of intelligence and presentation
• deterministic reporting structure
• consistent section layout
• executive-friendly language
• persistent report artifacts

Together with the telemetry loader and metrics engine, this module completes a pipeline capable of producing **transparent, operationally meaningful AI intelligence reports**.




In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

from config import MissionOrchestratorV3Config, MissionOrchestratorV3State


def _fmt_pct(value: Optional[float]) -> str:
    if value is None:
        return "N/A"
    return f"{value:.1f}%"


def _get_latest_run(mission_runs: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    if not mission_runs:
        return None
    # Assume runs are in chronological order in the dataset
    return mission_runs[-1]


def _build_mission_status_section(state: MissionOrchestratorV3State) -> str:
    status = state.get("mission_status_signal") or "UNKNOWN"
    data_quality = state.get("data_quality_signal") or "UNKNOWN"
    dq_details = state.get("data_quality_details") or ""
    mission_name = state.get("mission_name") or state.get("mission_id") or "Unknown Mission"
    return (
        f"## MISSION STATUS\n\n"
        f"Mission: {mission_name}\n\n"
        f"MISSION STATUS: {status}\n\n"
        f"DATA QUALITY: {data_quality}"
        + (f" ({dq_details})" if dq_details else "")
        + "\n"
    )


def _build_executive_summary_section(
    state: MissionOrchestratorV3State,
) -> str:
    mission_name = state.get("mission_name") or state.get("mission_id") or "Unknown Mission"
    eff = state.get("mission_efficiency_score")
    friction = state.get("operational_friction_score")
    reliability = state.get("agent_reliability_score")
    human_dep = state.get("human_dependency_index")

    mission_runs = state.get("mission_runs") or []
    latest_run = _get_latest_run(mission_runs)

    lines: List[str] = []
    lines.append("## EXECUTIVE SUMMARY\n")
    lines.append(f"Mission: {mission_name}\n")

    if eff is not None:
        lines.append(
            f"Performance is {_describe_efficiency(eff)} with a Mission Efficiency Score of {eff:.1f}."
        )
    else:
        lines.append("Performance efficiency cannot be scored yet (insufficient KPI configuration).")

    if friction is not None:
        lines.append(
            f"Operational friction is {_describe_friction(friction)} "
            f"(Operational Friction Score: {friction:.1f})."
        )

    if reliability is not None:
        lines.append(
            f"Agent reliability is {_describe_reliability(reliability)} "
            f"(Agent Reliability Score: {reliability:.1f})."
        )

    if human_dep is not None:
        lines.append(
            f"Human Dependency Index is {human_dep:.1f}, indicating "
            f"{_describe_human_dependency(human_dep)}."
        )

    if latest_run:
        kpis = latest_run.get("kpis") or {}
        if kpis:
            lines.append(f"Latest KPI snapshot: {kpis}.")

    return "\n".join(lines) + "\n"


def _describe_efficiency(score: float) -> str:
    if score >= 90.0:
        return "highly efficient"
    if score >= 75.0:
        return "acceptable"
    if score >= 60.0:
        return "needing improvement"
    return "underperforming"


def _describe_friction(score: float) -> str:
    if score <= 20.0:
        return "very smooth operations"
    if score <= 40.0:
        return "moderate friction"
    if score <= 60.0:
        return "high friction"
    return "severe bottlenecks"


def _describe_reliability(score: float) -> str:
    if score >= 90.0:
        return "highly reliable agents"
    if score >= 75.0:
        return "stable agents"
    if score >= 60.0:
        return "agents requiring monitoring"
    return "unreliable agents"


def _describe_human_dependency(score: float) -> str:
    if score <= 10.0:
        return "a fully or nearly fully automated system"
    if score <= 25.0:
        return "a mostly automated system with limited human oversight"
    if score <= 40.0:
        return "significant human involvement in mission workflows"
    return "a human-dependent system"


def _build_simple_trend_section(
    title: str,
    runs: List[Dict[str, Any]],
    kpi_key: Optional[str] = None,
) -> str:
    if not runs:
        return f"## {title}\n\nNo historical mission runs available.\n"

    lines: List[str] = [f"## {title}\n"]

    if kpi_key:
        series = [run.get("kpis", {}).get(kpi_key) for run in runs]
        series_clean = [v for v in series if isinstance(v, (int, float))]
        if series_clean:
            baseline = series_clean[0]
            latest = series_clean[-1]
            try:
                improvement = (baseline - latest) / baseline * 100.0
            except ZeroDivisionError:
                improvement = 0.0
            lines.append(f"Baseline: {baseline}")
            lines.append(f"Latest: {latest}")
            lines.append(f"Improvement vs baseline: {_fmt_pct(improvement)}")
        lines.append(f"Recent values: {series}")

    return "\n".join(lines) + "\n"


def build_report_sections(
    state: MissionOrchestratorV3State,
    config: MissionOrchestratorV3Config,
) -> MissionOrchestratorV3State:
    mission_runs = state.get("mission_runs") or []
    primary_kpi_name = config.default_primary_kpi

    state["report_mission_status"] = _build_mission_status_section(state)
    state["report_executive_summary"] = _build_executive_summary_section(state)
    state["report_mission_performance_trend"] = _build_simple_trend_section(
        "MISSION PERFORMANCE TREND",
        mission_runs,
        kpi_key=primary_kpi_name,
    )

    # For v3, keep the other sections simple and deterministic placeholders.
    state["report_bottleneck_analysis"] = "## BOTTLENECK ANALYSIS\n\nSee task_execution_logs for detailed bottleneck analysis.\n"
    state["report_risk_analysis"] = "## RISK ANALYSIS\n\nSee mission_risk_events for detailed risk breakdown.\n"
    state["report_agent_performance_analysis"] = "## AGENT PERFORMANCE ANALYSIS\n\nSee agent_performance_stats for per-agent metrics.\n"
    state["report_portfolio_health_summary"] = "## PORTFOLIO HEALTH SUMMARY\n\nSee mission_portfolio_snapshot for portfolio-level health.\n"

    return state


def assemble_report(
    state: MissionOrchestratorV3State,
    config: MissionOrchestratorV3Config,
    project_root: Optional[Path] = None,
) -> MissionOrchestratorV3State:
    """Assemble the final markdown report and save it to disk."""
    sections = [
        state.get("report_mission_status") or "",
        state.get("report_executive_summary") or "",
        state.get("report_mission_performance_trend") or "",
        state.get("report_bottleneck_analysis") or "",
        state.get("report_risk_analysis") or "",
        state.get("report_agent_performance_analysis") or "",
        state.get("report_portfolio_health_summary") or "",
    ]

    report = "\n\n".join(s.strip() for s in sections if s)
    state["mission_report"] = report

    root = project_root or Path(__file__).resolve().parents[4]
    reports_dir = Path(state.get("reports_dir") or config.reports_dir)
    output_dir = root / reports_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    mission_id = state.get("mission_id") or "mission"
    filename = f"mso_v3_report_{mission_id}_{timestamp}.md"
    output_path = output_dir / filename

    output_path.write_text(report, encoding="utf-8")
    state["report_file_path"] = str(output_path)

    return state

